# Tạo `searchable_text` bằng LLM

Notebook này đọc `merged_products_vi.csv`, làm sạch dữ liệu và dùng LLM để tạo văn bản `searchable_text` giàu ngữ nghĩa phục vụ semantic search + embedding + Qdrant.

In [ ]:
!pip -q install pandas openai

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI

INPUT_CSV = Path("merged_products_vi.csv")
OUTPUT_CSV = Path("merged_products_vi_with_searchable_text.csv")
OUTPUT_JSON = Path("merged_products_vi_with_searchable_text.json")

MODEL_NAME = "openai/gpt-oss-120b"
BASE_URL = "https://integrate.api.nvidia.com/v1"


def clean_text(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    cleaned = str(text).strip()
    if not cleaned:
        return ""
    cleaned = cleaned.replace("\r\n", "\n").replace("\r", "\n")
    cleaned = re.sub(r"\n+", "\n", cleaned)
    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    cleaned = re.sub(r"\s*\n\s*", "\n", cleaned)
    cleaned = re.sub(r"([,.;:!?])\1+", r"\1", cleaned)
    cleaned = re.sub(r"\s*([,.;:!?])\s*", r"\1 ", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned)
    return cleaned.strip(" \n\t,;")


def normalize_tags(tags):
    raw = clean_text(tags)
    if not raw:
        return ""
    parts = re.split(r"[|,;]+", raw)
    seen = set()
    out = []
    for part in parts:
        token = clean_text(part)
        if not token:
            continue
        key = token.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(token)
    return ", ".join(out)


def build_prompt_payload(row):
    return {
        "title": clean_text(row.get("title")),
        "brand": clean_text(row.get("brand")),
        "category": clean_text(row.get("category")),
        "description": clean_text(row.get("description")),
        "tags": normalize_tags(row.get("tags")),
    }


def fallback_searchable_text(payload):
    lines = []
    if payload["title"]:
        lines.append(f"Tên sản phẩm: {payload['title']}")
    if payload["brand"]:
        lines.append(f"Thương hiệu: {payload['brand']}")
    if payload["category"]:
        lines.append(f"Danh mục: {payload['category']}")
    if payload["description"]:
        desc = payload["description"]
        if len(desc) > 900:
            desc = desc[:900].rsplit(" ", 1)[0] + "..."
        lines.append(f"Mô tả: {desc}")
    if payload["tags"]:
        lines.append(f"Đặc điểm / tags: {payload['tags']}")
    return "\n".join(lines).strip()


def create_llm_client():
    api_key = ""

    # 1) Ưu tiên Colab Secrets (Google Colab -> Secrets -> NVIDIA_API_KEY)
    try:
        from google.colab import userdata  # type: ignore

        api_key = (userdata.get("NVIDIA_API_KEY") or "").strip()
    except Exception:
        pass

    # 2) Fallback sang biến môi trường
    if not api_key:
        api_key = os.environ.get("NVIDIA_API_KEY", "").strip()

    # 3) Cuối cùng cho nhập tay an toàn
    if not api_key:
        try:
            from getpass import getpass

            api_key = getpass("Nhập NVIDIA_API_KEY: ").strip()
        except Exception:
            api_key = ""

    if not api_key:
        raise ValueError(
            "Thiếu NVIDIA_API_KEY. Hãy thêm Colab Secret tên NVIDIA_API_KEY."
        )

    os.environ["NVIDIA_API_KEY"] = api_key
    return OpenAI(base_url=BASE_URL, api_key=api_key)


def build_searchable_text_with_llm(client, payload):
    system_prompt = (
        "Bạn là chuyên gia chuẩn hóa dữ liệu sản phẩm cho semantic search. "
        "Hãy tạo 1 đoạn searchable_text tiếng Việt, giàu ngữ nghĩa, dễ đọc, rõ ràng. "
        "Chỉ dùng các trường được cung cấp; không bịa thông tin. "
        "Nếu trường rỗng thì bỏ qua. "
        "Trả về text thuần, không markdown, không code block."
    )
    user_prompt = (
        "Dữ liệu đầu vào (JSON):\n"
        f"{json.dumps(payload, ensure_ascii=False)}\n\n"
        "Format mong muốn:\n"
        "Tên sản phẩm: ...\n"
        "Thương hiệu: ...\n"
        "Danh mục: ...\n"
        "Mô tả: ...\n"
        "Đặc điểm / tags: ...\n\n"
        "Lưu ý: Không ghi nhãn cho trường rỗng. Tổng độ dài lý tưởng <= 1200 ký tự."
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=900,
    )
    content = response.choices[0].message.content if response.choices else ""
    return clean_text(content)


def main(limit=None):
    if not INPUT_CSV.is_file():
        raise FileNotFoundError(f"Không tìm thấy file input: {INPUT_CSV}")

    df = pd.read_csv(INPUT_CSV)
    client = create_llm_client()

    if limit is not None:
        df = df.head(limit).copy()

    searchable_texts = []
    for _, row in df.iterrows():
        payload = build_prompt_payload(row)
        fallback_text = fallback_searchable_text(payload)
        try:
            llm_text = build_searchable_text_with_llm(client, payload)
            searchable_texts.append(llm_text or fallback_text)
        except Exception:
            searchable_texts.append(fallback_text)

    df["searchable_text"] = searchable_texts

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    df.to_json(OUTPUT_JSON, orient="records", force_ascii=False, indent=2)

    total_rows = len(df)
    empty_count = int((df["searchable_text"].str.strip() == "").sum())
    avg_len = float(df["searchable_text"].str.len().mean()) if total_rows else 0.0

    print(f"Tổng số dòng ban đầu: {total_rows}")
    print(f"Số dòng có searchable_text rỗng: {empty_count}")
    print(f"Độ dài trung bình searchable_text: {avg_len:.2f} ký tự")
    print("\n5 dòng mẫu (title + searchable_text):")

    sample = df[["title", "searchable_text"]].head(5)
    for i, r in sample.iterrows():
        print(f"\n[{i}] title: {clean_text(r['title']) or '(không có title)'}")
        print(r["searchable_text"] or "(rỗng)")

In [ ]:
# Chạy full dữ liệu: main()
# Chạy thử nhanh 50 dòng: main(limit=50)
main(limit=50)